In [1]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [2]:
# Create the agent
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [3]:
search = TavilySearch(max_results=2)
tools = [search]

In [4]:
llm_with_tools = model.bind_tools(tools)

sys_msg = SystemMessage(
    content="Use the tool if you need or just give me answer with your knowledge. Give short answers always")

# Node
def assistant(state: MessagesState) -> MessagesState:
   ai_message = llm_with_tools.invoke([sys_msg] + state["messages"])
   return {"messages": [ai_message]}


In [6]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")
builder.add_edge("assistant", END)


In [7]:
memory = InMemorySaver()
react_graph = builder.compile(checkpointer=memory)

# thread is identifies the session
config = {"configurable": {"thread_id": "1"}}

In [8]:
while True:
    prompt = input("Enter your prompt [q to quit]:")
    if prompt.lower() == 'q':
        break 
    human_message = HumanMessage(prompt)
    response = react_graph.invoke({"messages": [human_message]}, config=config)
    ai_message = response['messages'][-1]
    ai_message.pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Today (Wednesday, October 28th) in Vizag, the weather will be cloudy with a maximum temperature of 27°C and a minimum of 25°C. Winds will be around 16.7 km/h from the WNW, and humidity will be 90.5%. The UV index is 4.85.', 'extras': {'signature': 'CuUDAWkUfRNMGj+F+wTxFqwgGrHJnxa795m/PmtUXBO+l0YNL/Zf2SCCapHKNmX3CX1e+t5D1jsmD0LL+nel7pw3/+46eyzZakDJQMcOKwgJGrdW9Cb7uHtzInB3rBah/fM4rQrgrJjXTTfsfSMAcEb8LgJILuOYiDtGpoaw7xEQdtI14b5pNLJNeiblEg4sbJr8BZtcpJ/0Sww/J8XPs3R/Zg8LF17orsdklmHl5ZYlls/zhYF/xV0HxlpxvYsQCR0VZdz4sk/qEU2eGEZQlJZ1lqSJihcrGuFN8Wi5+64R6Er6QY2s7Cs4P5T4L/2CjW/e5CMpuuCee1Jy/Z4QHYU1TrckaG94dPg5TNodlLZxwSQnfCNqPMLhw6T0yRfw8bp/rxDphkGKf2Xm6kS0HgOEH439aZwrNKxCS7uKI10gBVbSyiGyJWlUMtdTiZ75jsKY4AzrkOw+j4GD+FEsT5M4xmpF9xDIcdeq1n+i7DLHZVBGK1q2St5ZE7xvft/x6BL9HEX5Rw3WjS++aCzXXQSGMk6JQHZshaE1ufjIAJA3Km+K1XQaPZ6KJ5caA/0pKybAB7tMosQ2C4eBg6S5DOK5wxqM/PR2iIdX1l4qDKe02WPeinW3Y4zYvGvBaoYT2s